# Full-data training: RNN, GRU, and LSTM

Choose a GPU runtime in Colab. Commit and push the current repository changes before running this notebook. It trains RNN, GRU, and LSTM at context lengths 3, 5, and 10: nine full-split runs. No sample caps are passed. Checkpoints and results are saved to your Google Drive.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/junaidjmomin/DeepLearning-RNNvsLSTMvsGRN.git'
REPO_DIR = Path('/content/DeepLearning-RNNvsLSTMvsGRN')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
assert (REPO_DIR / 'src/training/train_unified.py').is_file()
for name in ('train', 'validation', 'test', 'vocab'):
    assert (REPO_DIR / 'data/processed' / f'{name}.json').is_file(), name
print('Repository:', REPO_DIR)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow>=2.15,<2.22', 'numpy>=1.24,<3', 'pandas>=2.0'], check=True)
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime before full-data training.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/ContextNext_FullData')
ARTIFACT_DIR = OUTPUT_DIR / 'artifacts'
RESULTS_DIR = OUTPUT_DIR / 'results'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Saving to:', OUTPUT_DIR)

## Prepare contexts and train nine runs
Context 3 uses the tracked split files. Contexts 5 and 10 are generated from the same dialogue split seed. Each run uses every example in its matching split. A Colab session may expire; completed full-data runs are skipped when this cell is rerun.

In [ ]:
MODELS = ('rnn', 'gru', 'lstm')
CONTEXTS = (3, 5, 10)
for context_size in CONTEXTS:
    if context_size != 3:
        subprocess.run([sys.executable, '-m', 'src.data.preprocess', '--context-size', str(context_size)], cwd=REPO_DIR, check=True)
    for name in MODELS:
        suffix = '' if context_size == 3 else f'_ctx{context_size}'
        metrics_path = RESULTS_DIR / f'{name}{suffix}_test_metrics.json'
        checkpoint_path = ARTIFACT_DIR / 'models' / f'{name}{suffix}_best.keras'
        config_path = RESULTS_DIR / f'{name}{suffix}_config.json'
        if all(path.is_file() for path in (metrics_path, checkpoint_path, config_path)):
            config = json.loads(config_path.read_text())
            if config.get('sequence_length') == context_size and config.get('monitor') == 'val_top_1_accuracy' and all(config.get(key) is None for key in ('max_train_samples', 'max_validation_samples', 'max_test_samples')):
                print('Already trained on full splits:', name, context_size)
                continue
        print('Training:', name, 'context:', context_size, flush=True)
        subprocess.run([sys.executable, '-m', 'src.training.train_unified', '--model', name, '--sequence-length', str(context_size), '--monitor', 'val_top_1_accuracy',
                        '--artifact-dir', str(ARTIFACT_DIR), '--results-dir', str(RESULTS_DIR)],
                       cwd=REPO_DIR, check=True)

In [ ]:
import pandas as pd
rows = []
for context_size in CONTEXTS:
    for name in MODELS:
        suffix = '' if context_size == 3 else f'_ctx{context_size}'
        path = RESULTS_DIR / f'{name}{suffix}_test_metrics.json'
        if path.is_file():
            metrics = json.loads(path.read_text())
            rows.append({key: metrics.get(key) for key in ('model', 'sequence_length', 'samples', 'epochs_completed', 'top_1_accuracy', 'top_3_accuracy', 'top_5_accuracy', 'cross_entropy', 'perplexity')})
display(pd.DataFrame(rows))

Copy the Drive `artifacts/models/` and `results/` files into the matching local repository folders, then restart the API and dashboard. The files are ignored by Git because checkpoints can be large.